In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import os
from pandas.errors import EmptyDataError

# TEST phenotypes

In [2]:
dmeta = pd.read_csv("../DATA_intermediate/23_filter_EPN_indiv_metrics.tsv", sep="\t", header=0)
dmeta["POP_SITE"] = dmeta.apply(lambda x: str(x["POP"]) + "_" + x["SITE_ID"], axis=1)
dmeta = dmeta[["POP_SITE", "group"]].drop_duplicates().rename(columns={"group":"group_cluster"}).reset_index(drop=True)
print(dmeta.shape)
dmeta.head()

#dmeta[dmeta["group_cluster"] == "1530_AC"]
dmeta["group_cluster"].drop_duplicates()

(128, 2)


0     Central
3        East
4        West
27         ME
30         WI
Name: group_cluster, dtype: object

In [3]:
dpheno_ = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
dpheno = pd.merge(dpheno_, dmeta, on = "POP_SITE", how = "left")
dpheno = dpheno[dpheno['SET'] == 'TEST'].reset_index(drop=True)
###dpheno = dpheno[~dpheno['group_cluster'].isin(['ME','WI'])].reset_index(drop=True)
print(dpheno.shape)
dpheno["sample"] = dpheno["POP_SITE"]
dpheno.head()

(2048, 9)


,Trait_name,POP_SITE,POP_ID,SITE_ID,SET,mean,sd,n,group_cluster,sample
0,Average_Ring_Density,1329_CH,1329,CH,TEST,476.575417,45.597986,8,East,1329_CH
1,Average_Ring_Density,1329_ML,1329,ML,TEST,480.010872,108.572664,3,East,1329_ML
2,Average_Ring_Density,1528_ML,1528,ML,TEST,514.502734,62.179284,3,East,1528_ML
3,Average_Ring_Density,1530_AC,1530,AC,TEST,553.321396,48.783433,14,NaN,1530_AC
4,Average_Ring_Density,1530_CH,1530,CH,TEST,526.616667,93.195567,6,East,1530_CH


In [4]:
dpheno[dpheno["n"] < 3].shape

(48, 10)

In [5]:
dpheno = dpheno[dpheno['n']>2].reset_index(drop=True)
print(dpheno.shape)

(2000, 10)


# TRAIN phenotypes

In [6]:
tpheno_ = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
tpheno = pd.merge(tpheno_, dmeta, on = "POP_SITE", how = "left")
tpheno = tpheno[tpheno['SET'] == 'TRAIN'].reset_index(drop=True)
###tpheno = tpheno[~tpheno['group_cluster'].isin(['ME','WI'])].reset_index(drop=True)
print(tpheno.shape)
tpheno["sample"] = tpheno["POP_SITE"]

print(tpheno[tpheno["n"] < 3].shape)

tpheno = tpheno[tpheno['n']>2].reset_index(drop=True)
print(tpheno.shape)
tpheno.head()

(1738, 9)
(108, 10)
(1630, 10)


,Trait_name,POP_SITE,POP_ID,SITE_ID,SET,mean,sd,n,group_cluster,sample
0,Average_Ring_Density,1329_CH,1329,CH,TRAIN,465.670982,65.589447,7,East,1329_CH
1,Average_Ring_Density,1329_ML,1329,ML,TRAIN,513.398051,52.314507,13,East,1329_ML
2,Average_Ring_Density,1528_ML,1528,ML,TRAIN,487.240829,65.902345,7,East,1528_ML
3,Average_Ring_Density,1530_CH,1530,CH,TRAIN,518.358889,44.092654,6,East,1530_CH
4,Average_Ring_Density,1530_ML,1530,ML,TRAIN,474.392630,65.059481,10,East,1530_ML


# Combining genetic offset sets

In [7]:
list(range(2,43,2))

[2,
 4,
 6,
 8,
 10,
 12,
 14,
 16,
 18,
 20,
 22,
 24,
 26,
 28,
 30,
 32,
 34,
 36,
 38,
 40,
 42]

In [9]:
traits = ["Height", "Biomass_Increment", "Biomass_Increment_1980", "Biomass_Increment_1985", 
                "Biomass_Increment_1990", "Biomass_Increment_1995", "Biomass_Increment_2000", 
                "Biomass_Increment_2005", "Biomass_Increment_2010", "Biomass_Increment_2015",
                "Average_Ring_Density", "DBH", "Rs", "Rl", "Rr","Rc"]

gardens = ["PR","ML","CH","AC"]
#groups = ["East","Central","West"]
n_samples = [str(i) for i in range(2,43,2)]
iterations = ["1", "2", "3", "4", "5"]


In [10]:
D = []
M = []
for trait in traits:
    for garden in gardens:
        for n_sample in n_samples:
            for iteration in iterations:
                
                file_path = "../992_run_rda_exclude_pops/results/01_run_gradient_forest_" + garden + "_" + n_sample + "_" + iteration + "_" + trait + ".tsv"
                if os.path.exists(file_path):
                    try:
                        df_ = pd.read_csv(file_path, sep="\t", header=0)
                        df_["garden"] = garden
                        df_["marker"] = "1000"
                        df_["trait"] = trait
                        df_["n_samples"] = n_sample
                        df_["iteration"] = iteration
                        D.append(df_)
                    except EmptyDataError:
                        print("File is empty")
                        
    
                meta_path = "../992_run_rda_exclude_pops/samples/01_run_gradient_forest_TRAIN_subset_" + garden + "_" + n_sample + "_" + iteration + "_" + trait + ".tsv"
                if os.path.exists(meta_path):
                    try:
                        dm_ = pd.read_csv(meta_path, sep="\t", header=0)
                        dm_["garden"] = garden
                        dm_["marker"] = "1000"
                        dm_["trait"] = trait
                        dm_["n_samples"] = n_sample
                        dm_["iteration"] = iteration
                        M.append(dm_)
                    except EmptyDataError:
                        print("File is empty")

dD = pd.concat(D)
dM = pd.concat(M)
print(dD.shape)
dD.head()

File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File is empty
File i

,Trait_name,sample,SITE_ID,group,PC1,PC2,PC3,now_at1,now_at2,now_at3,new_at1,new_at2,new_at3,offset_rda,garden,marker,trait,n_samples,iteration
0,Height,2209_PR,PR,West,12.351204,-1.598833,0.851360,-1.472947,1.706775,8.348838,-0.966277,1.460752,4.831289,3.562358,PR,1000,Height,4,1
1,Height,325_PR,PR,Central,2.506557,-3.166880,1.958590,-1.386261,1.102182,4.913972,-0.966277,1.460752,4.831289,0.558387,PR,1000,Height,4,1
2,Height,4351_PR,PR,Central,-6.244846,1.531312,-1.399408,-0.425011,0.708292,0.903369,-0.966277,1.460752,4.831289,4.035805,PR,1000,Height,4,1
3,Height,4420_PR,PR,West,6.170760,3.960606,-2.183338,-0.264953,1.046679,3.837451,-0.966277,1.460752,4.831289,1.284922,PR,1000,Height,4,1
4,Height,6805_PR,PR,East,0.111999,-2.575310,0.678848,-1.410570,1.376089,5.140428,-0.966277,1.460752,4.831289,0.547842,PR,1000,Height,4,1


In [11]:
print(len(M), len(D))
dM.head()

4295 3975


,sample,POP_ID,Trait_name,SITE_ID,group,PC1,PC2,PC3,garden,marker,trait,n_samples,iteration
0,6964_PR,6964,Height,PR,Central,-1.056001,4.261677,0.595031,PR,1000,Height,2,1
1,6965_PR,6965,Height,PR,West,4.551163,0.376039,-0.186452,PR,1000,Height,2,1
0,325_PR,325,Height,PR,Central,2.506557,-3.166880,1.958590,PR,1000,Height,2,2
1,6965_PR,6965,Height,PR,West,4.551163,0.376039,-0.186452,PR,1000,Height,2,2
0,6916_PR,6916,Height,PR,Central,1.648885,0.218255,2.387171,PR,1000,Height,2,3


In [12]:
dD.head()

,Trait_name,sample,SITE_ID,group,PC1,PC2,PC3,now_at1,now_at2,now_at3,new_at1,new_at2,new_at3,offset_rda,garden,marker,trait,n_samples,iteration
0,Height,2209_PR,PR,West,12.351204,-1.598833,0.851360,-1.472947,1.706775,8.348838,-0.966277,1.460752,4.831289,3.562358,PR,1000,Height,4,1
1,Height,325_PR,PR,Central,2.506557,-3.166880,1.958590,-1.386261,1.102182,4.913972,-0.966277,1.460752,4.831289,0.558387,PR,1000,Height,4,1
2,Height,4351_PR,PR,Central,-6.244846,1.531312,-1.399408,-0.425011,0.708292,0.903369,-0.966277,1.460752,4.831289,4.035805,PR,1000,Height,4,1
3,Height,4420_PR,PR,West,6.170760,3.960606,-2.183338,-0.264953,1.046679,3.837451,-0.966277,1.460752,4.831289,1.284922,PR,1000,Height,4,1
4,Height,6805_PR,PR,East,0.111999,-2.575310,0.678848,-1.410570,1.376089,5.140428,-0.966277,1.460752,4.831289,0.547842,PR,1000,Height,4,1


In [13]:
dD_sub = dD[["sample","SITE_ID","garden","offset_rda","marker","trait","n_samples","iteration","group"]].reset_index(drop=True)
print(dD_sub.shape)
dD_sub.head()

(135330, 9)


,sample,SITE_ID,garden,offset_rda,marker,trait,n_samples,iteration,group
0,2209_PR,PR,PR,3.562358,1000,Height,4,1,West
1,325_PR,PR,PR,0.558387,1000,Height,4,1,Central
2,4351_PR,PR,PR,4.035805,1000,Height,4,1,Central
3,4420_PR,PR,PR,1.284922,1000,Height,4,1,West
4,6805_PR,PR,PR,0.547842,1000,Height,4,1,East


In [14]:
dM_sub = dM[["sample","trait","n_samples","iteration","group"]].reset_index(drop=True)
print(dM_sub.shape)
dM_sub.head()

(74680, 5)


,sample,trait,n_samples,iteration,group
0,6964_PR,Height,2,1,Central
1,6965_PR,Height,2,1,West
2,325_PR,Height,2,2,Central
3,6965_PR,Height,2,2,West
4,6916_PR,Height,2,3,Central


### Test

In [15]:
dD_merge = pd.merge(dD_sub, dpheno, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(dD_merge.shape)
dD_merge.head()

(135330, 17)


,sample,SITE_ID,garden,offset_rda,marker,trait,n_samples,iteration,group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group_cluster
0,2209_PR,PR,PR,3.562358,1000,Height,4,1,West,Height,2209_PR,2209.0,TEST,980.525258,302.158899,4.0,West
1,325_PR,PR,PR,0.558387,1000,Height,4,1,Central,Height,325_PR,325.0,TEST,1239.891145,121.743290,8.0,Central
2,4351_PR,PR,PR,4.035805,1000,Height,4,1,Central,Height,4351_PR,4351.0,TEST,1423.598605,68.068593,4.0,Central
3,4420_PR,PR,PR,1.284922,1000,Height,4,1,West,Height,4420_PR,4420.0,TEST,678.153897,154.272486,7.0,West
4,6805_PR,PR,PR,0.547842,1000,Height,4,1,East,Height,6805_PR,6805.0,TEST,1353.974749,229.855027,7.0,East


### Train

In [16]:
# Subsetting populations selected in the train
dT_sub = pd.merge(dM_sub, dD_sub, on = ["sample","trait","n_samples","iteration","group"], how = "left")
print(dD_sub.shape, dT_sub.shape)

(135330, 9) (74680, 9)


In [17]:
tD_merge = pd.merge(dT_sub, tpheno, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(tD_merge.shape)
tD_merge = tD_merge.dropna().reset_index(drop=True)
print(tD_merge.shape)
tD_merge.tail()

(74680, 17)
(70187, 17)


,sample,trait,n_samples,iteration,group,SITE_ID,garden,offset_rda,marker,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group_cluster
70182,4277_AC,Rc,10,5,Central,AC,AC,0.180235,1000,Rc,4277_AC,4277.0,TRAIN,0.891376,0.055955,7.0,Central
70183,4351_AC,Rc,10,5,Central,AC,AC,0.162068,1000,Rc,4351_AC,4351.0,TRAIN,0.829416,0.067626,10.0,Central
70184,6941_AC,Rc,10,5,Central,AC,AC,0.161407,1000,Rc,6941_AC,6941.0,TRAIN,0.931869,0.072256,13.0,Central
70185,6969_AC,Rc,10,5,West,AC,AC,0.584844,1000,Rc,6969_AC,6969.0,TRAIN,0.916375,0.103709,5.0,West
70186,6983_AC,Rc,10,5,West,AC,AC,0.505147,1000,Rc,6983_AC,6983.0,TRAIN,0.884334,0.111582,8.0,West


### Calculating rho

In [18]:
def spearman_group(df, col_a, col_b, group_cols):
    results = []
    for name, g in df.groupby(group_cols):
        if len(g) < 3:
            rho, p = np.nan, np.nan
        elif ((len(g[col_a].dropna())<3) or (len(g[col_b].dropna())<3)):
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(g[col_a], g[col_b], nan_policy='omit')
        # name is tuple if multiple group columns
        name = name if isinstance(name, tuple) else (name,)
        results.append((*name, rho, p, len(g)))
            
    res_df = pd.DataFrame(results, columns=list(group_cols) + ['spearman_rho', 'p_value', 'n_samps'])
    
    # Adjust p-values
    mask = res_df['p_value'].notna()
    res_df['p_adj'] = np.nan
    if mask.any():
        res_df.loc[mask, 'p_adj'] = multipletests(res_df.loc[mask, 'p_value'], method='fdr_bh')[1]

    return res_df

### Test

In [19]:
df_cors = spearman_group(dD_merge, 'offset_rda', 'mean', ['SITE_ID','trait','marker','n_samples','iteration'])
df_cors['sign'] = df_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
df_cors["SET"] = "TEST"
df_cors.head()

,SITE_ID,trait,marker,n_samples,iteration,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,-0.766532,4.962015e-07,32,0.000164,minus,TEST
1,AC,Average_Ring_Density,1000,10,2,-0.766532,4.962015e-07,32,0.000164,minus,TEST
2,AC,Average_Ring_Density,1000,10,3,-0.766532,4.962015e-07,32,0.000164,minus,TEST
3,AC,Average_Ring_Density,1000,10,4,-0.766532,4.962015e-07,32,0.000164,minus,TEST
4,AC,Average_Ring_Density,1000,10,5,-0.766532,4.962015e-07,32,0.000164,minus,TEST


### Train

In [20]:
dt_cors = spearman_group(tD_merge, 'offset_rda', 'mean', ['SITE_ID','trait','marker','n_samples','iteration'])
dt_cors['sign'] = dt_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
dt_cors["SET"] = "TRAIN"
dt_cors.head()

,SITE_ID,trait,marker,n_samples,iteration,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,-0.716667,0.029818,9,0.171377,minus,TRAIN
1,AC,Average_Ring_Density,1000,10,2,-0.716667,0.029818,9,0.171377,minus,TRAIN
2,AC,Average_Ring_Density,1000,10,3,-0.716667,0.029818,9,0.171377,minus,TRAIN
3,AC,Average_Ring_Density,1000,10,4,-0.716667,0.029818,9,0.171377,minus,TRAIN
4,AC,Average_Ring_Density,1000,10,5,-0.716667,0.029818,9,0.171377,minus,TRAIN


### Combined

In [21]:
dcombined = pd.concat([df_cors, dt_cors], ignore_index=True)
dcombined.head()

,SITE_ID,trait,marker,n_samples,iteration,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,-0.766532,4.962015e-07,32,0.000164,minus,TEST
1,AC,Average_Ring_Density,1000,10,2,-0.766532,4.962015e-07,32,0.000164,minus,TEST
2,AC,Average_Ring_Density,1000,10,3,-0.766532,4.962015e-07,32,0.000164,minus,TEST
3,AC,Average_Ring_Density,1000,10,4,-0.766532,4.962015e-07,32,0.000164,minus,TEST
4,AC,Average_Ring_Density,1000,10,5,-0.766532,4.962015e-07,32,0.000164,minus,TEST


In [22]:
dcombined.to_csv("03_garden_pops_rda.tsv", sep="\t", header=True, index=False)
dcombined.to_excel("03_garden_pops_rda.xlsx")

# With clusters

### Test

In [23]:
df_cors = spearman_group(dD_merge, 'offset_rda', 'mean', ['SITE_ID','trait','marker','n_samples','iteration','group'])
df_cors['sign'] = df_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
df_cors["SET"] = "TEST"
df_cors.head()

,SITE_ID,trait,marker,n_samples,iteration,group,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,Central,-0.573427,0.051266,12,0.431945,minus,TEST
1,AC,Average_Ring_Density,1000,10,1,East,-0.466667,0.173939,10,0.754619,minus,TEST
2,AC,Average_Ring_Density,1000,10,1,ME,NaN,NaN,1,NaN,minus,TEST
3,AC,Average_Ring_Density,1000,10,1,WI,NaN,NaN,1,NaN,minus,TEST
4,AC,Average_Ring_Density,1000,10,1,West,-0.200000,0.747060,6,0.898061,minus,TEST


### Train

In [24]:
dt_cors = spearman_group(tD_merge, 'offset_rda', 'mean', ['SITE_ID','trait','marker','n_samples','iteration','group'])
dt_cors['sign'] = dt_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
dt_cors["SET"] = "TRAIN"
dt_cors.head()

,SITE_ID,trait,marker,n_samples,iteration,group,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,Central,0.1,0.872889,5,0.951982,plus,TRAIN
1,AC,Average_Ring_Density,1000,10,1,East,NaN,NaN,2,NaN,minus,TRAIN
2,AC,Average_Ring_Density,1000,10,1,West,NaN,NaN,2,NaN,minus,TRAIN
3,AC,Average_Ring_Density,1000,10,2,Central,0.1,0.872889,5,0.951982,plus,TRAIN
4,AC,Average_Ring_Density,1000,10,2,East,NaN,NaN,2,NaN,minus,TRAIN


In [25]:
dcombined2 = pd.concat([df_cors, dt_cors], ignore_index=True)
dcombined2.head()

,SITE_ID,trait,marker,n_samples,iteration,group,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,Central,-0.573427,0.051266,12,0.431945,minus,TEST
1,AC,Average_Ring_Density,1000,10,1,East,-0.466667,0.173939,10,0.754619,minus,TEST
2,AC,Average_Ring_Density,1000,10,1,ME,NaN,NaN,1,NaN,minus,TEST
3,AC,Average_Ring_Density,1000,10,1,WI,NaN,NaN,1,NaN,minus,TEST
4,AC,Average_Ring_Density,1000,10,1,West,-0.200000,0.747060,6,0.898061,minus,TEST


In [26]:
dcombined2.to_csv("03_garden_pops_rda_clusters.tsv", sep="\t", header=True, index=False)

# END